# Sync vs Async Endpoints

This notebook covers:

1. Choose between `def` and `async def` endpoints based on workload
2. Measure latency and throughput of each against a simulated downstream
3. Understand how FastAPI runs sync endpoints in a threadpool
4. Recognize that `async` is not automatically faster

**Scope**: FastAPI + `httpx.AsyncClient` against the in-process app via `ASGITransport`. No external network.

The single most common newcomer question: "should I use `def` or `async def`?" The answer depends on **where the time goes**. This notebook makes that visible.

## 1. The Two Endpoint Modes

A FastAPI route handler is either:

- **`async def`** — a coroutine. FastAPI awaits it directly on the event loop.
- **`def`** — a regular function. FastAPI runs it in a threadpool (Starlette's anyio worker pool, ~40 threads by default) so it doesn't block the loop.

Both work. The choice changes the runtime cost model:

| Mode      | Runs on            | Best for                                              |
|-----------|--------------------|-------------------------------------------------------|
| `async def` | Event loop         | I/O with async libs (`httpx.AsyncClient`, asyncpg, redis.asyncio) |
| `def`       | Threadpool worker | Sync I/O libs (`requests`, `psycopg2`, file I/O)      |

The rule: if your handler calls anything that takes time, that thing must be either **awaitable** (and you `await` it inside `async def`) or **sync** (and the whole handler is `def`). Mixing — calling a blocking sync function from `async def` without offloading it — is the disaster in notebook 3.2.

## 2. How FastAPI Schedules Sync vs Async

Under the hood, Starlette inspects each route's handler when the app starts:

- If the handler is a coroutine function, Starlette `await`s it on the loop.
- If it's a plain function, Starlette wraps each invocation in `anyio.to_thread.run_sync(...)` — so 100 concurrent requests to the sync handler will use up to ~40 threadpool slots and queue the rest.

Two implications:

1. **A sync endpoint is not "slow"**, it's just bounded by the threadpool. If 1000 requests hit a slow sync endpoint, the 41st request waits.
2. **An async endpoint that secretly blocks** (calls `time.sleep` or sync DB drivers) holds the entire event loop hostage. *All* requests in flight stall, not just the ones to that endpoint.

## 3. A Simulated Slow Downstream

To make the difference visible we need a workload that *waits* — that's what most real handlers do (database query, HTTP call to another service). We'll simulate with `await asyncio.sleep(0.1)` in the async case and `time.sleep(0.1)` in the sync case.

100 ms is enough to be measurable but small enough that the whole benchmark runs in seconds.

In [ ]:
import asyncio, time
from fastapi import FastAPI
import httpx

LATENCY_S = 0.1  # simulated downstream wait per request

app = FastAPI()

@app.get("/sync")
def sync_endpoint():
    # def => runs on threadpool. time.sleep blocks the worker thread, not the loop.
    time.sleep(LATENCY_S)
    return {"mode": "sync"}

@app.get("/async-correct")
async def async_correct():
    # async def + awaitable sleep => yields the loop while waiting.
    await asyncio.sleep(LATENCY_S)
    return {"mode": "async-correct"}

print("two handlers registered: /sync (def), /async-correct (async def)")

## 4. Benchmark: Latency Under Concurrency

To exercise the app from inside a notebook with real concurrency, we use `httpx.AsyncClient` wired to an `ASGITransport` that targets the FastAPI app in-process. This is the same trick we'll use in notebook 7.2 for async tests.

The benchmark fires N requests concurrently and reports total wall time and effective throughput. With per-request latency `L` and concurrency `N`:

- **Serial** (loop blocked): `total ≈ N * L`
- **Concurrent** (loop free or threadpool ample): `total ≈ L`

In [ ]:
from httpx import AsyncClient, ASGITransport

N = 50  # concurrent requests

async def bench(path: str, n: int = N) -> float:
    transport = ASGITransport(app=app)
    async with AsyncClient(transport=transport, base_url="http://test") as ac:
        await ac.get(path)  # warm-up so we measure steady state, not first-request setup
        start = time.perf_counter()
        await asyncio.gather(*(ac.get(path) for _ in range(n)))
        return time.perf_counter() - start

async def main():
    for path in ["/sync", "/async-correct"]:
        t = await bench(path)
        print(f"{path:18} {N:>3} concurrent -> {t*1000:7.1f} ms total  ({N/t:6.1f} req/s)")

asyncio.run(main())

print(f"\nideal serial would be ~{N * LATENCY_S * 1000:.0f} ms; ideal fully-concurrent ~{LATENCY_S * 1000:.0f} ms.")

## 5. When Async Helps (I/O Bound)

The benchmark above shows async + `await asyncio.sleep` finishing all 50 requests in ~100 ms — the same as one request — because every coroutine yields the loop while waiting. The loop juggles all 50 in parallel.

Real-world equivalent: an endpoint that calls another HTTP service via `httpx.AsyncClient`, a database query via `asyncpg`, or a Redis lookup via `redis.asyncio`. All of these `await` while the OS does the actual waiting. The loop schedules other requests during that wait.

**The async win compounds**: a single uvicorn worker can hold thousands of in-flight requests if each spends most of its time waiting on I/O. A threaded sync server bounded by 40 threads tops out at 40 concurrent waits, regardless of how cheap the wait is.

In [ ]:
# I/O-bound async pattern: fan out to multiple downstreams in parallel within one request.
# `fetch_one` stands in for a downstream HTTP / DB call — in production it would
# `await client.get(...)` against a shared async client.
async def fetch_one(key: str) -> dict:
    await asyncio.sleep(LATENCY_S)
    return {"key": key, "value": key.upper()}

@app.get("/lookup")
async def lookup():
    # 5 downstreams in parallel. Total wait ≈ max(latencies), not sum.
    results = await asyncio.gather(*(fetch_one(k) for k in ["a", "b", "c", "d", "e"]))
    return {"results": results}

async def run_one():
    async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as ac:
        await ac.get("/lookup")  # warm-up
        start = time.perf_counter()
        r = await ac.get("/lookup")
        return r.json(), time.perf_counter() - start

body, t = asyncio.run(run_one())
print(f"5 parallel downstreams -> {t*1000:.1f} ms (ideal: {LATENCY_S*1000:.0f} ms, serial would be {5*LATENCY_S*1000:.0f} ms)")
print("body:", body)

## 6. When Sync Is Fine (or Better)

`async def` is not a free upgrade. Pick `def` when:

- **The libraries you call are sync**. `psycopg2`, `requests`, `sqlite3`, most file I/O. Wrapping them in `asyncio.to_thread` inside `async def` works but adds ceremony for no win — `def` is simpler.
- **Handlers are CPU-bound and short**. Hashing a small payload, running a Pydantic validation. Async doesn't help; the loop wants you off it anyway. (Long CPU-bound work needs `ProcessPoolExecutor` — see notebook 3.2.)
- **The team isn't comfortable with async yet**. A `def` handler that doesn't block is easier to read and harder to break than an `async def` someone might accidentally introduce a sync call into.

The disaster case — `async def` that calls a sync blocking function without offloading — is what notebook 3.2 is about. Once you know that pitfall, the choice is mostly about which client libraries you're using.

## Key Takeaways

- **`async def` runs on the event loop.** Use it when you have something to `await` — an async client, an async DB driver, `asyncio.sleep`.
- **`def` runs on the threadpool.** Starlette wraps it in `anyio.to_thread.run_sync`. Bounded by ~40 threads.
- **The win from async is concurrency under I/O wait.** A single worker can hold thousands of in-flight requests if they're all waiting on the network.
- **Async is not automatically faster.** For CPU-bound or sync-library handlers, `def` is simpler and equivalent.
- **The one rule that matters**: inside `async def`, never call a blocking sync function without offloading it. That's notebook 3.2.

## Exercises

**1. Convert and measure.** Take this sync handler and write the async equivalent:

```python
@app.get("/quote/{ticker}")
def get_quote(ticker: str):
    time.sleep(0.05)  # pretend HTTP call to a price service
    return {"ticker": ticker, "price": 100.0}
```

Run the benchmark from §4 against both at `N=50`. Expect both to finish in ~100–200 ms — the sync one is fine because 50 < threadpool size. Now retry with `N=100` and explain what changes.

**2. Parallel downstreams.** Write an endpoint `GET /portfolio/quotes?tickers=AAPL&tickers=MSFT&tickers=GOOGL` that fans out to a `fetch_quote(ticker)` coroutine (use `asyncio.sleep(0.1)` as the stand-in) and returns the merged result. Total handler time should be ~100 ms, not 300 ms.

**3. Spot the regression.** A teammate "optimizes" this:
```python
@app.get("/")
async def root():
    return {"db": query_db()}  # query_db is sync, uses psycopg2
```
What's wrong? What's the cheapest fix without rewriting `query_db`?